# Reprodução — Alkhatib et al. (2021), SOME/IP IDS com RNN

1. **Verificar os modelos publicados** (reproduz a Tabela VII do paper, ~98%).
2. **Treinar o RNN do zero** (PyTorch).
3. **Resultados finais**: métricas + matriz de confusão + curvas ROC/AUC.

Funciona local (Jupyter) ou no Colab (descomente a célula de clone).


## 0. Setup


In [ ]:
# No Colab: clone o repo e entre nele (ajuste a URL do seu remoto).
# !git clone https://github.com/SEU_USUARIO/Alkhatib2021-repro.git
# %cd Alkhatib2021-repro

import os, sys
assert os.path.isdir('src') and os.path.isdir('data'), 'Rode a partir da raiz do repo (src/ e data/)'
sys.path.insert(0, '.')
print('cwd OK:', os.getcwd())


In [ ]:
!pip -q install numpy scikit-learn h5py torch matplotlib pandas
print('deps OK')


## 1. Verificar os modelos publicados (reproduz a Tabela VII)


In [ ]:
from src.verify_published import main as verify_published
verify_published()


## 2. Treinar o RNN do zero (PyTorch)

Hiperparâmetros do paper (Adam, lr 1e-3, batch 100) + **gradient clipping**.
Em GPU roda em segundos; em CPU, poucos minutos.


In [ ]:
from src.train import run
model, metrics = run(epochs=60)


## 3. Resultados finais — métricas, matriz de confusão e curvas ROC

Por padrão usa o **melhor modelo publicado** (reproduz as figuras do paper).
Para usar o modelo que você treinou, troque `SOURCE = 'trained'`.


In [ ]:
SOURCE = 'published'   # 'published' (modelos do paper) ou 'trained' (seu treino)

import numpy as np, glob, os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             accuracy_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from src import data as datamod
from src.data import CLASS_NAMES

_, _, X_test, y_test = datamod.load()

if SOURCE == 'published':
    from src.verify_published import load_weights, forward
    name = sorted(glob.glob('data/published_models/*.h5'))[-1]
    logits = forward(X_test, load_weights(name))
    title = os.path.basename(name)
else:
    import torch
    from src.model import AlkhatibRNN
    m = AlkhatibRNN(); m.load_state_dict(torch.load('results/rnn_pytorch.pt')); m.eval()
    with torch.no_grad():
        logits = m(torch.from_numpy(X_test)).numpy()
    title = 'RNN PyTorch (treino do zero)'

# softmax -> probabilidades
probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
y_pred = probs.argmax(1)


### 3a. Métricas por classe


In [ ]:
p, r, f1, sup = precision_recall_fscore_support(y_test, y_pred, labels=range(5), zero_division=0)
df = pd.DataFrame({'Precision': p, 'Recall': r, 'F1': f1, 'Support': sup},
                  index=CLASS_NAMES).round(3)
print(f'Modelo: {title}')
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
df


### 3b. Matriz de confusão + curvas ROC (one-vs-rest, com AUC)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- matriz de confusão ---
cm = confusion_matrix(y_test, y_pred, labels=range(5))
im = ax1.imshow(cm, cmap='Blues')
ax1.set_xticks(range(5)); ax1.set_yticks(range(5))
ax1.set_xticklabels(CLASS_NAMES, rotation=45, ha='right'); ax1.set_yticklabels(CLASS_NAMES)
ax1.set_xlabel('Predito'); ax1.set_ylabel('Verdadeiro'); ax1.set_title(f'Matriz de Confusão\n{title}')
for i in range(5):
    for j in range(5):
        ax1.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                 color='white' if cm[i, j] > cm.max()/2 else 'black')

# --- curvas ROC ---
Yb = label_binarize(y_test, classes=range(5))
for i, cname in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(Yb[:, i], probs[:, i])
    ax2.plot(fpr, tpr, lw=1.5, label=f'{cname} (AUC={auc(fpr, tpr):.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.7)
ax2.set_xlabel('Taxa de Falso Positivo'); ax2.set_ylabel('Taxa de Verdadeiro Positivo')
ax2.set_title('Curvas ROC por classe'); ax2.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()


---
**Esperado:** modelo publicado ~98% acc, AUC perto de 1 (reproduz o paper);
treino do zero ~97%. As classes minoritárias (ErrorOnError) são as mais difíceis.
